# Generate SituatiONION Controlled Triples

This notebook creates 200 base, paraphrase, and counterfactual triples for SituatiONION v3. Each counterfactual changes exactly one situation-graph field, while each paraphrase preserves the graph. It also writes template-held-out train/test JSONL files.

The default data is a validated starting benchmark, not a final research dataset. Before reporting results, inspect the generated CSV and expand to more event families and surface templates.

In [ ]:
from itertools import permutations
from pathlib import Path
import json

import pandas as pd

N_PER_TEMPLATE = 20
OUTPUT_DIR = Path('data')
OUTPUT_DIR.mkdir(exist_ok=True)
PEOPLE = ['Maya', 'Leo', 'Nora', 'Omar', 'Iris', 'Jules', 'Ava', 'Kai', 'Mina', 'Theo', 'Ravi', 'Uma', 'Sam', 'Lena', 'Noah', 'Zoe']
PAIRS = list(permutations(PEOPLE, 2))
print(f'Generating {5 * 2 * N_PER_TEMPLATE} triples from {len(PAIRS)} entity pairs.')

In [ ]:
def situation_graph(agent, recipient, event, object_name, polarity='affirmed', cause=None, time_relation=None, topic=None):
    return {
        'agent': agent, 'recipient': recipient, 'event': event, 'object': object_name,
        'polarity': polarity, 'cause': cause, 'time_relation': time_relation, 'topic': topic,
    }

def other_person(person, offset):
    return PEOPLE[(PEOPLE.index(person) + offset) % len(PEOPLE)]

def changed_fields(base_graph, counter_graph):
    return {name for name in base_graph if base_graph[name] != counter_graph[name]}

def row(identifier, template_id, changed_field, base, paraphrase, counterfactual, base_graph, counter_graph):
    return {
        'id': identifier, 'template_id': template_id, 'changed_field': changed_field,
        'base': base, 'paraphrase': paraphrase, 'counterfactual': counterfactual,
        'base_graph': base_graph, 'paraphrase_graph': base_graph.copy(), 'counterfactual_graph': counter_graph,
    }

In [ ]:
def make_example(kind, style, agent, recipient, index):
    other_agent = other_person(agent, index + 1)
    other_recipient = other_person(recipient, index + 2)

    if kind == 'agent':
        base_graph = situation_graph(agent, recipient, 'give', 'key', cause='locked_out', topic='restore_access')
        counter_graph = situation_graph(other_agent, recipient, 'give', 'key', cause='locked_out', topic='restore_access')
        if style == 0:
            base = f'{recipient} was locked out, so {agent} gave {recipient} the key.'
            paraphrase = f'Because {recipient} could not enter, {agent} handed the key to {recipient}.'
            counter = f'{recipient} was locked out, so {other_agent} gave {recipient} the key.'
        else:
            base = f'When {recipient} could not get inside, {agent} passed the key to {recipient}.'
            paraphrase = f'{agent} handed {recipient} a key after learning that {recipient} was locked out.'
            counter = f'When {recipient} could not get inside, {other_agent} passed the key to {recipient}.'

    elif kind == 'recipient':
        base_graph = situation_graph(agent, recipient, 'give', 'map', cause='lost', topic='navigation_help')
        counter_graph = situation_graph(agent, other_recipient, 'give', 'map', cause='lost', topic='navigation_help')
        if style == 0:
            base = f'{recipient} was lost, so {agent} gave {recipient} a map.'
            paraphrase = f'Because {recipient} was lost, {agent} handed a map to {recipient}.'
            counter = f'{other_recipient} was lost, so {agent} gave {other_recipient} a map.'
        else:
            base = f'{agent} offered {recipient} a map after {recipient} got lost.'
            paraphrase = f'After {recipient} lost the way, {agent} passed a map to {recipient}.'
            counter = f'{agent} offered {other_recipient} a map after {other_recipient} got lost.'

    elif kind == 'polarity':
        base_graph = situation_graph(agent, recipient, 'repair', 'bike', cause='broken', topic='repair')
        counter_graph = situation_graph(agent, recipient, 'repair', 'bike', polarity='negated', cause='broken', topic='repair')
        if style == 0:
            base = f'{agent} repaired the bike because it was broken.'
            paraphrase = f'Because the bike was broken, {agent} fixed it.'
            counter = f'{agent} did not repair the bike even though it was broken.'
        else:
            base = f'After noticing the broken bike, {agent} fixed it.'
            paraphrase = f'{agent} saw that the bike was broken and repaired it.'
            counter = f'After noticing the broken bike, {agent} chose not to fix it.'

    elif kind == 'cause':
        base_graph = situation_graph(agent, recipient, 'give', 'map', cause='lost', topic='navigation_help')
        counter_graph = situation_graph(agent, recipient, 'give', 'map', cause='curious', topic='navigation_help')
        if style == 0:
            base = f'{agent} gave {recipient} a map because {recipient} was lost.'
            paraphrase = f'Because {recipient} was lost, {agent} handed over a map.'
            counter = f'{agent} gave {recipient} a map because {recipient} was curious.'
        else:
            base = f'Knowing that {recipient} was lost, {agent} offered {recipient} a map.'
            paraphrase = f'{agent} offered a map to {recipient} after learning that {recipient} had lost the way.'
            counter = f'Knowing that {recipient} was curious, {agent} offered {recipient} a map.'

    elif kind == 'time_relation':
        base_graph = situation_graph(agent, recipient, 'call', 'phone_call', time_relation='after', topic='communication')
        counter_graph = situation_graph(agent, recipient, 'call', 'phone_call', time_relation='before', topic='communication')
        if style == 0:
            base = f'{agent} called {recipient} after the storm ended.'
            paraphrase = f'After the storm ended, {agent} phoned {recipient}.'
            counter = f'{agent} called {recipient} before the storm ended.'
        else:
            base = f'Once the storm was over, {agent} rang {recipient}.'
            paraphrase = f'{agent} phoned {recipient} when the storm had ended.'
            counter = f'Before the storm was over, {agent} rang {recipient}.'

    else:
        raise ValueError(f'Unknown family: {kind}')

    return row(f'{kind}_{style}_{index:03d}', f'{kind}_{style}', kind, base, paraphrase, counter, base_graph, counter_graph)

FAMILIES = ['agent', 'recipient', 'polarity', 'cause', 'time_relation']

In [ ]:
def build_dataset():
    rows = []
    for family_index, family in enumerate(FAMILIES):
        for style in range(2):
            start = (family_index * 47 + style * N_PER_TEMPLATE) % len(PAIRS)
            for local_index in range(N_PER_TEMPLATE):
                agent, recipient = PAIRS[(start + local_index) % len(PAIRS)]
                item = make_example(family, style, agent, recipient, family_index * 100 + style * N_PER_TEMPLATE + local_index)
                item['split'] = 'train' if style == 0 else 'test'
                rows.append(item)
    return rows

def validate(rows):
    assert len(rows) == len(FAMILIES) * 2 * N_PER_TEMPLATE
    assert len({item['id'] for item in rows}) == len(rows)
    for item in rows:
        assert item['base'] != item['paraphrase'] != item['counterfactual']
        assert item['base_graph'] == item['paraphrase_graph']
        assert changed_fields(item['base_graph'], item['counterfactual_graph']) == {item['changed_field']}

ROWS = build_dataset()
validate(ROWS)
print(f'Validated {len(ROWS)} triples.')
pd.DataFrame([{key: item[key] for key in ['id', 'template_id', 'split', 'changed_field', 'base', 'counterfactual']} for item in ROWS]).head()

In [ ]:
def write_jsonl(path, rows):
    with path.open('w', encoding='utf-8') as handle:
        for item in rows:
            handle.write(json.dumps(item, ensure_ascii=False) + '\n')

write_jsonl(OUTPUT_DIR / 'situationion_triples_v1.jsonl', ROWS)
write_jsonl(OUTPUT_DIR / 'situationion_train_v1.jsonl', [item for item in ROWS if item['split'] == 'train'])
write_jsonl(OUTPUT_DIR / 'situationion_test_v1.jsonl', [item for item in ROWS if item['split'] == 'test'])

flat_rows = []
for item in ROWS:
    flat = {key: item[key] for key in ['id', 'template_id', 'split', 'changed_field', 'base', 'paraphrase', 'counterfactual']}
    flat.update({f'base_{key}': value for key, value in item['base_graph'].items()})
    flat.update({f'counterfactual_{key}': value for key, value in item['counterfactual_graph'].items()})
    flat_rows.append(flat)
pd.DataFrame(flat_rows).to_csv(OUTPUT_DIR / 'situationion_triples_v1.csv', index=False)

display(pd.DataFrame(flat_rows).groupby(['split', 'changed_field']).size())
print('Wrote JSONL and CSV files under data/.')

## Next steps

Audit samples from every template group in the CSV. Then add more event families and at least four to six paraphrase templates per family. Keep entire templates in a held-out split; otherwise a probe can learn wording instead of situation structure.